# Model Comparison

Compare logistic regression, random forests, and gradient boosting on the same validation weeks.


## Setup and Local Inputs

Load the files exported by notebook 04. No cloud queries or test data are used.


In [16]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import db_dtypes  # Register BigQuery date types stored in the Parquet metadata.
from IPython.display import display
from threadpoolctl import threadpool_limits
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss

project_root = Path.cwd().resolve()
if not (project_root / "dbt_promo_abuse").is_dir():
    project_root = project_root.parent
comparison_dir = project_root / "data" / "processed" / "model_comparison"
required_files = ["train.parquet", "validation.parquet", "logistic_baseline.joblib", "feature_columns.json"]
assert all((comparison_dir / f).exists() for f in required_files), (
    "Run the final export cell in notebook 04 first."
)
train_df = pd.read_parquet(comparison_dir / "train.parquet")
validation_df = pd.read_parquet(comparison_dir / "validation.parquet")
feature_columns = json.loads((comparison_dir / "feature_columns.json").read_text())
# Load only the trusted model exported locally by notebook 04.
logistic_model = joblib.load(comparison_dir / "logistic_baseline.joblib")
assert len(feature_columns) == len(set(feature_columns)) == 31
assert not set(feature_columns) & {"user_id", "prediction_date", "abuse_next_7d", "dataset_split"}
for frame, split, start, end in [
    (train_df, "train", "2016-02-01", "2016-04-25"),
    (validation_df, "validation", "2016-05-02", "2016-05-16"),
]:
    assert frame["dataset_split"].eq(split).all()
    assert frame[["user_id", "prediction_date"]].notna().all().all()
    assert not frame.duplicated(["user_id", "prediction_date"]).any()
    assert frame["abuse_next_7d"].isin([0, 1]).all()
    assert set(pd.to_datetime(frame["prediction_date"])) == set(pd.date_range(start, end, freq="7D"))
assert len(train_df) == 483431 and train_df["abuse_next_7d"].sum() == 5552
assert len(validation_df) == 102860 and validation_df["abuse_next_7d"].sum() == 1149
X_train = train_df[feature_columns].astype("float64")
y_train = train_df["abuse_next_7d"].astype("int64")
X_validation = validation_df[feature_columns].astype("float64")
y_validation = validation_df["abuse_next_7d"].astype("int64")
recency_columns = [c for c in feature_columns if c.startswith("days_since_")]
ratio_columns = ["offline_redemption_rate_30d"]
complete_columns = [c for c in feature_columns if c not in recency_columns + ratio_columns]
for features in [X_train, X_validation]:
    assert not np.isinf(features.to_numpy()).any()
    assert not features[complete_columns].isna().any().any()
print(f"Ready: {len(X_train):,} training rows, {len(X_validation):,} validation rows, {len(feature_columns)} predictors.")


def evaluate_comparison(comparison_models, X_validation, validation_df, y_validation, fit_times):
    comparison_scores = {}
    metric_rows, weekly_rows = [], []
    for name, model in comparison_models.items():
        scores = model.predict_proba(X_validation)[:, 1]
        comparison_scores[name] = scores
        metric_rows.append({
            "model": name,
            "average_precision": average_precision_score(y_validation, scores),
            "roc_auc": roc_auc_score(y_validation, scores),
            "brier_score": brier_score_loss(y_validation, scores),
            "log_loss": log_loss(y_validation, scores, labels=[0, 1]),
            "fit_seconds": fit_times[name],
        })
        scored = validation_df[["user_id", "prediction_date", "abuse_next_7d"]].copy()
        scored["risk_score"] = scores
        for date, week in scored.groupby("prediction_date"):
            ranked = week.sort_values(["risk_score", "user_id"], ascending=[False, True], kind="mergesort")
            positives = int(week["abuse_next_7d"].sum())
            for k in [100, 250, 500]:
                reviewed = ranked.head(k)
                tp = int(reviewed["abuse_next_7d"].sum())
                weekly_rows.append({
                    "model": name, "prediction_date": date, "capacity": k,
                    "reviewed": len(reviewed), "true_positives": tp,
                    "false_positives": len(reviewed) - tp, "weekly_positives": positives,
                    "precision": tp / len(reviewed),
                    "recall": tp / positives if positives else np.nan,
                })
    model_comparison = pd.DataFrame(metric_rows).sort_values("average_precision", ascending=False)
    weekly_comparison = pd.DataFrame(weekly_rows)
    capacity_comparison = weekly_comparison.groupby(["model", "capacity"]).agg(
        reviewed=("reviewed", "sum"), true_positives=("true_positives", "sum"),
        false_positives=("false_positives", "sum"), positives=("weekly_positives", "sum"),
    )
    capacity_comparison["precision"] = capacity_comparison["true_positives"] / capacity_comparison["reviewed"]
    capacity_comparison["recall"] = capacity_comparison["true_positives"] / capacity_comparison["positives"].replace(0, np.nan)
    display(model_comparison.round(5))
    display(capacity_comparison.round(4))
    print("Weekly results at the illustrative capacity of 250:")
    display(weekly_comparison.loc[weekly_comparison["capacity"].eq(250)].round(4))
    return comparison_scores, model_comparison, weekly_comparison, capacity_comparison


def inspect_calibration(comparison_scores, y_validation):
    calibration_frames = []
    for name, scores in comparison_scores.items():
        scored = pd.DataFrame({"risk_score": scores, "target": y_validation.to_numpy()})
        scored["risk_band"] = pd.cut(
            scored["risk_score"], [0, 0.01, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0],
            include_lowest=True,
        )
        table = scored.groupby("risk_band", observed=True).agg(
            observations=("target", "size"), positives=("target", "sum"),
            mean_predicted=("risk_score", "mean"), observed_rate=("target", "mean"),
        ).reset_index()
        table.insert(0, "model", name)
        table["overprediction_pp"] = (table["mean_predicted"] - table["observed_rate"]) * 100
        calibration_frames.append(table)
    forest_calibration_comparison = pd.concat(calibration_frames, ignore_index=True)
    with pd.option_context("display.max_rows", None):
        display(forest_calibration_comparison.round(4))
    return forest_calibration_comparison


Ready: 483,431 training rows, 102,860 validation rows, 31 predictors.


## Fit Two Forests

Compare depth 12 with leaves of at least 20 rows against depth 20 with leaves of at least 10 rows. Use 200 trees each and two CPU workers.


In [9]:
forest_settings = {
    "Forest depth12 leaf20": {"max_depth": 12, "min_samples_leaf": 20},
    "Forest depth20 leaf10": {"max_depth": 20, "min_samples_leaf": 10},
}
comparison_models = {"Logistic baseline": logistic_model}
fit_times = {"Logistic baseline": np.nan}  # Already fitted in notebook 04.
for name, settings in forest_settings.items():
    # Same missing-value meanings as logistic regression; trees need no scaling.
    preprocessing = ColumnTransformer([
        ("recency", SimpleImputer(strategy="constant", fill_value=31), recency_columns),
        ("ratio", SimpleImputer(strategy="constant", fill_value=0, add_indicator=True), ratio_columns),
        ("complete", "passthrough", complete_columns),
    ], remainder="drop")
    model = Pipeline([
        ("preprocess", preprocessing),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_features="sqrt",
            bootstrap=True,
            max_samples=0.8,
            class_weight=None,
            random_state=42,
            n_jobs=2,
            **settings,
        )),
    ])
    print(f"Fitting {name}...", flush=True)
    started = time.perf_counter()
    model.fit(X_train, y_train)
    fit_times[name] = time.perf_counter() - started
    comparison_models[name] = model
    print(f"Finished in {fit_times[name]:.1f} seconds", flush=True)


Fitting Forest depth12 leaf20...
Finished in 17.8 seconds
Fitting Forest depth20 leaf10...
Finished in 18.8 seconds


## Ranking and Weekly Review Results

Compare probability quality, ranking, and fixed weekly review capacities using identical validation observations.


In [13]:
comparison_scores, model_comparison, weekly_comparison, capacity_comparison = evaluate_comparison(
    comparison_models, X_validation, validation_df, y_validation, fit_times
)


,model,average_precision,roc_auc,brier_score,log_loss,fit_seconds
5,Forest depth30 leaf10,0.36974,0.95019,0.00873,0.03707,19.43745
2,Forest depth20 leaf10,0.36938,0.95093,0.00873,0.03677,18.76275
4,Forest depth20 leaf20,0.35606,0.95018,0.00886,0.03705,18.46984
3,Forest depth12 leaf10,0.33676,0.94723,0.00896,0.03789,16.86578
1,Forest depth12 leaf20,0.33168,0.94677,0.00903,0.03814,17.81644
0,Logistic baseline,0.17376,0.90998,0.01017,0.04490,NaN


reviewed  true_positives  false_positives  \
model                 capacity                                              
Forest depth12 leaf10 100            300             194              106   
                      250            750             332              418   
                      500           1500             466             1034   
Forest depth12 leaf20 100            300             188              112   
                      250            750             338              412   
                      500           1500             463             1037   
Forest depth20 leaf10 100            300             201               99   
                      250            750             358              392   
                      500           1500             497             1003   
Forest depth20 leaf20 100            300             199              101   
                      250            750             345              405   
                      500           1500             486             1014   
Forest depth30 leaf10 100            300             208               92   
                      250            750             359              391   
                      500           1500             503              997   
Logistic baseline     100            300             112              188   
                      250            750             212              538   
                      500           1500             322             1178   

                                positives  precision  recall  
model                 capacity                                
Forest depth12 leaf10 100            1149     0.6467  0.1688  
                      250            1149     0.4427  0.2889  
                      500            1149     0.3107  0.4056  
Forest depth12 leaf20 100            1149     0.6267  0.1636  
                      250            1149     0.4507  0.2942  
                      500            1149     0.3087  0.4030  
Forest depth20 leaf10 100            1149     0.6700  0.1749  
                      250            1149     0.4773  0.3116  
                      500            1149     0.3313  0.4326  
Forest depth20 leaf20 100            1149     0.6633  0.1732  
                      250            1149     0.4600  0.3003  
                      500            1149     0.3240  0.4230  
Forest depth30 leaf10 100            1149     0.6933  0.1810  
                      250            1149     0.4787  0.3124  
                      500            1149     0.3353  0.4378  
Logistic baseline     100            1149     0.3733  0.0975  
                      250            1149     0.2827  0.1845  
                      500            1149     0.2147  0.2802

Weekly results at the illustrative capacity of 250:


,model,prediction_date,capacity,reviewed,true_positives,false_positives,weekly_positives,precision,recall
1,Logistic baseline,2016-05-02,250,250,60,190,371,0.240,0.1617
4,Logistic baseline,2016-05-09,250,250,75,175,385,0.300,0.1948
7,Logistic baseline,2016-05-16,250,250,77,173,393,0.308,0.1959
10,Forest depth12 leaf20,2016-05-02,250,250,114,136,371,0.456,0.3073
13,Forest depth12 leaf20,2016-05-09,250,250,121,129,385,0.484,0.3143
16,Forest depth12 leaf20,2016-05-16,250,250,103,147,393,0.412,0.2621
19,Forest depth20 leaf10,2016-05-02,250,250,114,136,371,0.456,0.3073
22,Forest depth20 leaf10,2016-05-09,250,250,125,125,385,0.500,0.3247
25,Forest depth20 leaf10,2016-05-16,250,250,119,131,393,0.476,0.3028
28,Forest depth12 leaf10,2016-05-02,250,250,112,138,371,0.448,0.3019


## Probability Calibration

Compare predicted and observed risks in fixed probability bands for each model.


In [14]:
forest_calibration_comparison = inspect_calibration(comparison_scores, y_validation)


,model,risk_band,observations,positives,mean_predicted,observed_rate,overprediction_pp
0,Logistic baseline,"(-0.001, 0.01]",76217,106,0.0024,0.0014,0.0960
1,Logistic baseline,"(0.01, 0.05]",21728,421,0.0213,0.0194,0.1968
2,Logistic baseline,"(0.05, 0.1]",2913,241,0.0687,0.0827,-1.4002
3,Logistic baseline,"(0.1, 0.2]",1220,166,0.1358,0.1361,-0.0252
4,Logistic baseline,"(0.2, 0.4]",480,100,0.2783,0.2083,6.9939
5,Logistic baseline,"(0.4, 0.6]",167,49,0.4906,0.2934,19.7213
6,Logistic baseline,"(0.6, 0.8]",91,42,0.6987,0.4615,23.7198
7,Logistic baseline,"(0.8, 1.0]",44,24,0.8964,0.5455,35.0916
8,Forest depth12 leaf20,"(-0.001, 0.01]",77769,53,0.0017,0.0007,0.0973
9,Forest depth12 leaf20,"(0.01, 0.05]",18897,283,0.0227,0.0150,0.7701


## Focused Forest Refinement

Vary depth and minimum leaf size separately, then rerun the existing evaluation cells for all candidates.


In [12]:
# Complete the depth/leaf-size comparison and test one deeper candidate.
# Keep trees, sample fraction, feature sampling, seed, and weighting fixed.
refinement_settings = {
    "Forest depth12 leaf10": {"max_depth": 12, "min_samples_leaf": 10},
    "Forest depth20 leaf20": {"max_depth": 20, "min_samples_leaf": 20},
    "Forest depth30 leaf10": {"max_depth": 30, "min_samples_leaf": 10},
}
assert "Forest depth20 leaf10" in comparison_models, "Fit the initial forests first."
for name, settings in refinement_settings.items():
    candidate = clone(comparison_models["Forest depth20 leaf10"]).set_params(
        **{f"model__{key}": value for key, value in settings.items()}
    )
    print(f"Fitting {name}...", flush=True)
    started = time.perf_counter()
    candidate.fit(X_train, y_train)
    fit_times[name] = time.perf_counter() - started
    comparison_models[name] = candidate
    print(f"Finished in {fit_times[name]:.1f} seconds", flush=True)

print("Now rerun Ranking and Weekly Review Results, then Probability Calibration above.")
print("Those cells will evaluate all six models using the same validation observations.")


Fitting Forest depth12 leaf10...
Finished in 16.9 seconds
Fitting Forest depth20 leaf20...
Finished in 18.5 seconds
Fitting Forest depth30 leaf10...
Finished in 19.4 seconds
Now rerun Ranking and Weekly Review Results, then Probability Calibration above.
Those cells will evaluate all six models using the same validation observations.


## Histogram Gradient Boosting

Fit two boosted-tree candidates with 15 or 31 leaves, fixed 200 iterations, and training-only preprocessing.


In [17]:
assert "Forest depth20 leaf10" in comparison_models, "Run the initial forest fit first."
boosting_settings = {
    "Boosting leaves15": 15,
    "Boosting leaves31": 31,
}
for name, leaf_nodes in boosting_settings.items():
    # Match the forest's missing-value handling and the same 31 source features.
    model = Pipeline([
        ("preprocess", clone(comparison_models["Forest depth20 leaf10"].named_steps["preprocess"])),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=200,
            max_leaf_nodes=leaf_nodes,
            min_samples_leaf=30,
            l2_regularization=1.0,
            early_stopping=False,
            class_weight=None,
            random_state=42,
        )),
    ])
    # Fixed iterations avoid an automatic random validation split inside training.
    print(f"Fitting {name}...", flush=True)
    started = time.perf_counter()
    with threadpool_limits(limits=2):
        model.fit(X_train, y_train)
    fit_times[name] = time.perf_counter() - started
    comparison_models[name] = model
    print(f"Finished {model.named_steps['model'].n_iter_} iterations in {fit_times[name]:.1f} seconds", flush=True)


Fitting Boosting leaves15...
Finished 200 iterations in 10.2 seconds
Fitting Boosting leaves31...
Finished 200 iterations in 10.4 seconds


## Updated Model Comparison

Evaluate all fitted candidates with the same metrics, weekly capacities, and calibration bands.


In [18]:
with threadpool_limits(limits=2):
    comparison_scores, model_comparison, weekly_comparison, capacity_comparison = evaluate_comparison(
        comparison_models, X_validation, validation_df, y_validation, fit_times
    )
    forest_calibration_comparison = inspect_calibration(comparison_scores, y_validation)


,model,average_precision,roc_auc,brier_score,log_loss,fit_seconds
7,Boosting leaves31,0.40242,0.95814,0.00842,0.03454,10.41978
6,Boosting leaves15,0.38712,0.95567,0.00853,0.03527,10.15526
5,Forest depth30 leaf10,0.36974,0.95019,0.00873,0.03707,19.43745
2,Forest depth20 leaf10,0.36938,0.95093,0.00873,0.03677,18.76275
4,Forest depth20 leaf20,0.35606,0.95018,0.00886,0.03705,18.46984
3,Forest depth12 leaf10,0.33676,0.94723,0.00896,0.03789,16.86578
1,Forest depth12 leaf20,0.33168,0.94677,0.00903,0.03814,17.81644
0,Logistic baseline,0.17376,0.90998,0.01017,0.04490,NaN


reviewed  true_positives  false_positives  \
model                 capacity                                              
Boosting leaves15     100            300             214               86   
                      250            750             371              379   
                      500           1500             511              989   
Boosting leaves31     100            300             219               81   
                      250            750             387              363   
                      500           1500             530              970   
Forest depth12 leaf10 100            300             194              106   
                      250            750             332              418   
                      500           1500             466             1034   
Forest depth12 leaf20 100            300             188              112   
                      250            750             338              412   
                      500           1500             463             1037   
Forest depth20 leaf10 100            300             201               99   
                      250            750             358              392   
                      500           1500             497             1003   
Forest depth20 leaf20 100            300             199              101   
                      250            750             345              405   
                      500           1500             486             1014   
Forest depth30 leaf10 100            300             208               92   
                      250            750             359              391   
                      500           1500             503              997   
Logistic baseline     100            300             112              188   
                      250            750             212              538   
                      500           1500             322             1178   

                                positives  precision  recall  
model                 capacity                                
Boosting leaves15     100            1149     0.7133  0.1862  
                      250            1149     0.4947  0.3229  
                      500            1149     0.3407  0.4447  
Boosting leaves31     100            1149     0.7300  0.1906  
                      250            1149     0.5160  0.3368  
                      500            1149     0.3533  0.4613  
Forest depth12 leaf10 100            1149     0.6467  0.1688  
                      250            1149     0.4427  0.2889  
                      500            1149     0.3107  0.4056  
Forest depth12 leaf20 100            1149     0.6267  0.1636  
                      250            1149     0.4507  0.2942  
                      500            1149     0.3087  0.4030  
Forest depth20 leaf10 100            1149     0.6700  0.1749  
                      250            1149     0.4773  0.3116  
                      500            1149     0.3313  0.4326  
Forest depth20 leaf20 100            1149     0.6633  0.1732  
                      250            1149     0.4600  0.3003  
                      500            1149     0.3240  0.4230  
Forest depth30 leaf10 100            1149     0.6933  0.1810  
                      250            1149     0.4787  0.3124  
                      500            1149     0.3353  0.4378  
Logistic baseline     100            1149     0.3733  0.0975  
                      250            1149     0.2827  0.1845  
                      500            1149     0.2147  0.2802

Weekly results at the illustrative capacity of 250:


,model,prediction_date,capacity,reviewed,true_positives,false_positives,weekly_positives,precision,recall
1,Logistic baseline,2016-05-02,250,250,60,190,371,0.240,0.1617
4,Logistic baseline,2016-05-09,250,250,75,175,385,0.300,0.1948
7,Logistic baseline,2016-05-16,250,250,77,173,393,0.308,0.1959
10,Forest depth12 leaf20,2016-05-02,250,250,114,136,371,0.456,0.3073
13,Forest depth12 leaf20,2016-05-09,250,250,121,129,385,0.484,0.3143
16,Forest depth12 leaf20,2016-05-16,250,250,103,147,393,0.412,0.2621
19,Forest depth20 leaf10,2016-05-02,250,250,114,136,371,0.456,0.3073
22,Forest depth20 leaf10,2016-05-09,250,250,125,125,385,0.500,0.3247
25,Forest depth20 leaf10,2016-05-16,250,250,119,131,393,0.476,0.3028
28,Forest depth12 leaf10,2016-05-02,250,250,112,138,371,0.448,0.3019


,model,risk_band,observations,positives,mean_predicted,observed_rate,overprediction_pp
0,Logistic baseline,"(-0.001, 0.01]",76217,106,0.0024,0.0014,0.0960
1,Logistic baseline,"(0.01, 0.05]",21728,421,0.0213,0.0194,0.1968
2,Logistic baseline,"(0.05, 0.1]",2913,241,0.0687,0.0827,-1.4002
3,Logistic baseline,"(0.1, 0.2]",1220,166,0.1358,0.1361,-0.0252
4,Logistic baseline,"(0.2, 0.4]",480,100,0.2783,0.2083,6.9939
5,Logistic baseline,"(0.4, 0.6]",167,49,0.4906,0.2934,19.7213
6,Logistic baseline,"(0.6, 0.8]",91,42,0.6987,0.4615,23.7198
7,Logistic baseline,"(0.8, 1.0]",44,24,0.8964,0.5455,35.0916
8,Forest depth12 leaf20,"(-0.001, 0.01]",77769,53,0.0017,0.0007,0.0973
9,Forest depth12 leaf20,"(0.01, 0.05]",18897,283,0.0227,0.0150,0.7701


## User-Level Uncertainty for Validation Ranking

Estimate validation metric uncertainty by resampling users with all their validation weeks together.


In [19]:
# Resample users, carrying all of each sampled user's validation weeks together.
# Sample weights represent how many times each user was drawn in a bootstrap replicate.
bootstrap_models = ["Logistic baseline", "Forest depth20 leaf10", "Boosting leaves31"]
assert all(name in comparison_scores for name in bootstrap_models)
user_codes, validation_users = pd.factorize(validation_df["user_id"], sort=False)
n_validation_users = len(validation_users)
user_row_counts = np.bincount(user_codes)
assert n_validation_users > 1 and user_row_counts.sum() == len(validation_df)

bootstrap_replicates = 1000
rng = np.random.default_rng(2026)
bootstrap_metrics = np.full((bootstrap_replicates, len(bootstrap_models), 3), np.nan)
for b in range(bootstrap_replicates):
    sampled_user_counts = np.bincount(
        rng.integers(0, n_validation_users, size=n_validation_users),
        minlength=n_validation_users,
    )
    row_weights = sampled_user_counts[user_codes]
    sampled_rows = row_weights > 0
    for j, name in enumerate(bootstrap_models):
        scores = comparison_scores[name][sampled_rows]
        targets = y_validation.to_numpy()[sampled_rows]
        weights = row_weights[sampled_rows]
        bootstrap_metrics[b, j, 0] = average_precision_score(
            targets, scores, sample_weight=weights
        )
        bootstrap_metrics[b, j, 1] = roc_auc_score(
            targets, scores, sample_weight=weights
        )
        bootstrap_metrics[b, j, 2] = brier_score_loss(
            targets, scores, sample_weight=weights
        )

metric_names = ["average_precision", "roc_auc", "brier_score"]
bootstrap_summary_rows = []
for j, name in enumerate(bootstrap_models):
    point_scores = comparison_scores[name]
    point = [
        average_precision_score(y_validation, point_scores),
        roc_auc_score(y_validation, point_scores),
        brier_score_loss(y_validation, point_scores),
    ]
    for k, metric in enumerate(metric_names):
        low, high = np.quantile(bootstrap_metrics[:, j, k], [0.025, 0.975])
        bootstrap_summary_rows.append({
            "model": name, "metric": metric, "estimate": point[k],
            "user_cluster_ci_95_low": low, "user_cluster_ci_95_high": high,
        })

boost_idx = bootstrap_models.index("Boosting leaves31")
forest_idx = bootstrap_models.index("Forest depth20 leaf10")
logistic_idx = bootstrap_models.index("Logistic baseline")
for other_idx, other_name in [(forest_idx, "Forest depth20 leaf10"),
                              (logistic_idx, "Logistic baseline")]:
    paired_ap_delta = bootstrap_metrics[:, boost_idx, 0] - bootstrap_metrics[:, other_idx, 0]
    low, high = np.quantile(paired_ap_delta, [0.025, 0.975])
    point_delta = (
        average_precision_score(y_validation, comparison_scores["Boosting leaves31"])
        - average_precision_score(y_validation, comparison_scores[other_name])
    )
    bootstrap_summary_rows.append({
        "model": f"Boosting leaves31 minus {other_name}",
        "metric": "average_precision_difference",
        "estimate": point_delta,
        "user_cluster_ci_95_low": low,
        "user_cluster_ci_95_high": high,
    })

user_cluster_bootstrap_summary = pd.DataFrame(bootstrap_summary_rows)
print(f"Resampled {n_validation_users:,} distinct users, preserving their validation weeks; {bootstrap_replicates:,} replicates.")
display(user_cluster_bootstrap_summary.round(5))
print("These intervals describe validation sampling uncertainty; model selection on these same weeks can still make the comparison optimistic.")


Resampled 51,054 distinct users, preserving their validation weeks; 1,000 replicates.


,model,metric,estimate,user_cluster_ci_95_low,user_cluster_ci_95_high
0,Logistic baseline,average_precision,0.17376,0.15600,0.19469
1,Logistic baseline,roc_auc,0.90998,0.90163,0.91727
2,Logistic baseline,brier_score,0.01017,0.00965,0.01076
3,Forest depth20 leaf10,average_precision,0.36938,0.34056,0.39632
4,Forest depth20 leaf10,roc_auc,0.95093,0.94608,0.95521
5,Forest depth20 leaf10,brier_score,0.00873,0.00829,0.00920
6,Boosting leaves31,average_precision,0.40242,0.37279,0.43019
7,Boosting leaves31,roc_auc,0.95814,0.95357,0.96209
8,Boosting leaves31,brier_score,0.00842,0.00798,0.00889
9,Boosting leaves31 minus Forest depth20 leaf10,average_precision_difference,0.03304,0.02161,0.04479


These intervals describe validation sampling uncertainty; model selection on these same weeks can still make the comparison optimistic.
